# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahmoud-Beram/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*



**Research question:** Given position, content age, and content properties, can we
rank FlyRank client pages by how much CTR "opportunity" they are missing — to help
editors prioritize a limited weekly review budget?

**Decision this supports:** which content pages an editor reviews and refreshes first,
given they can only handle a limited number per week (out of tens of thousands of
eligible pages per client base).

**Cost of a wrong call:** flagging a page that isn't really underperforming wastes an
editor's limited time; missing a genuinely declining page lets it keep losing traffic
unnoticed.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*


**Source:** FlyRank internship-warehouse (Hugging Face), `fact_content_daily_performance`
(month=2026-03) joined with `dim_content` on `content_hash_id`.

**Size & time window:** 61,911 unique (client, content) pairs with ≥500 impressions,
from a single month (March 2026), across 36 unique clients.

**An initial iteration** also incorporated 90-day query-mix features
(`fact_content_query_90d`, window Apr 2–Jun 30, 2026), but these were identified as a
temporal leak during validation (Section 3) and removed entirely from the final
feature set.

**Exclusions:**
- `is_published = TRUE AND is_deleted = FALSE` (drafts/archived pages excluded).
- `total_imp >= 500` (insufficient impressions can't give a stable CTR estimate).
- Missing `word_count`/`search_volume`/etc. imputed with the column median (not
  zero-filled, to avoid injecting a false "zero content" signal).

**Privacy:** all client and content identifiers are pseudonymous hashes
(`client_hash_id`, `content_hash_id`), used only for joining/grouping — never as
features. No raw client names, URLs, or search queries appear anywhere in this work.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*



**1. Ground Truth Definition (The Label):**
To evaluate model performance, we needed a discrete label for "underperforming" content. Since CTR decays exponentially with ranking position, comparing absolute CTRs is invalid. Instead, we bucketed articles into groups of 10 positions (e.g., pos 1-10, 11-20) and defined `truly_underperforming` as articles falling in the bottom 25% (Q1) of CTR *within their specific position bucket*.

**2. Model Selection & Features:**
We deployed a **Random Forest Regressor** to predict the expected CTR based on a clean feature set: `avg_pos`, `word_count`, `search_volume`, `competition`, `cpc`, `backlinks`, and `content_age_days`. The gap between the predicted expected CTR and the actual CTR (multiplied by impressions) forms the `lost_clicks` queue.

**3. Validation Design & The Cold-Start Limit:**
We rigorously audited the model using multiple splitting strategies:
- **Random Split:** Yielded an overly optimistic R² (0.23), masking client-specific overfitting.
- **GroupKFold (Client Holdout):** When holding out entire clients, the R² dropped below zero (-0.03). This confirmed a strict business rule: the model cannot generalize to brand-new clients (Cold-Start Problem). It requires historical client data to establish a baseline.

**4. Leakage Audit:**
During our initial iteration, we included features from `fact_content_query_90d` (e.g., `rare_traffic_percent`). A temporal leakage audit revealed that this 90-day window overlapped with the future relative to our March 2026 snapshot. These features artificially inflated performance and were strictly removed to maintain an honest evaluation.

**5. A Crucial Methodological Caveat (The LR Illusion):**
During our audit, Linear Regression unexpectedly outperformed Random Forest in Ranking Precision (Precision@K) despite having a near-zero R². A deep dive into the coefficients revealed a structural bias: the Linear Regression decision was almost entirely driven by `avg_pos` (its coefficient magnitude was ~2.3x the next feature, essentially ignoring content properties). Because our ground truth label (`truly_underperforming`) was itself constructed using `avg_pos` buckets, Linear Regression's higher Precision@K was an artifact of aligning with the evaluation label's construction, not a superior understanding of the market. Consequently, Random Forest was retained as the deployed model to ensure all content features (age, volume, etc.) influence the final ranking.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 4. Results: Model vs. Baseline

We evaluated the Random Forest model against a simple heuristic baseline (the Week-4 rule) and the dataset's natural base rate. To simulate real-world conditions and avoid data leakage, we utilized a 5-Fold `GroupKFold` cross-validation strategy (grouping by `client_hash_id`).

The primary evaluation metric was **Precision@50** (the percentage of truly underperforming articles found within the top 50 recommendations per fold).

**Performance Summary (Precision@50 across 5 Folds):**
- **Base Rate (Random Guessing):** 20.3% (± 6.2%)
- **Baseline (Week-4 Rule):** 26.4% (± 8.6%)
- **Final Model (Random Forest):** 46.4% (± 10.4%)

**Key Takeaway:**
The Random Forest model significantly outperforms both random guessing and the simple heuristic rule. By upgrading from the baseline rule to the ML model, the content team's hit rate increases from 26.4% to 46.4% (an approximate **1.75x improvement** in efficiency). This means out of every 30 articles reviewed by an editor, the model successfully identifies ~14 genuinely failing articles, compared to only ~8 using the old rule.


## 5. Limitations

*What this work cannot claim.*

##Limitations & Honest Framing

To ensure this model is used safely and effectively, we must explicitly state what it **cannot** do:

**1. The Cold-Start Limitation (No New Clients):**
The model is incapable of generalizing to brand-new clients with no Google Search Console history (demonstrated by a negative R² under GroupKFold client-holdout validation). The model relies heavily on historical data to calibrate what "normal" performance looks like for a specific domain. Therefore, this queue should **never** be generated for clients with less than 30 days of data.

**2. Decision-Support, Not Full Automation:**
This model is highly directional; it flags mathematical anomalies (missed opportunities) but does not diagnose the root cause. For example, an article might have a low CTR because Google is displaying a "Zero-Click" Answer Box, not because our title is poorly written. Therefore, the model's output must remain a prioritization queue for human editors. It should **never** be used to automatically overwrite titles or delete content without manual SERP verification.

**3. Evaluation Bias Vulnerability:**
As noted in the methodology, evaluating ranking systems on position-derived targets (like our `truly_underperforming` label) inherently favors models that over-index on position (like Linear Regression). While Random Forest mitigates this by utilizing the broader feature set, measuring true business impact ultimately requires A/B testing the actual CTR lift post-refresh, rather than relying solely on offline Precision metrics.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 6. Ranked Recommendations

To ensure practical usability for the editorial team, the model outputs a strict prioritization queue ranked by `lost_clicks` (Expected CTR minus Actual CTR, multiplied by Impressions). This focuses human effort on the largest missed-click opportunities.

Matching the Week-4 baseline's streamlined design, every flagged page in the queue carries a unified directive:
- **Reason Code:** `CTR_BELOW_MODEL_EXPECTATION`
- **Action:** `SNIPPET_FIX`

**The Workflow (The "So What"):**
When an editor works through the top of the queue, the required action is to evaluate and rewrite the Title Tag and Meta Description. Because the model successfully identifies pages that hold strong ranking positions but fail to convert impressions into clicks, a targeted "Snippet Fix" is the most direct and lowest-cost intervention to recover the estimated lost traffic.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## 7. Reproducibility

To ensure transparency and allow independent verification of our findings, the complete end-to-end pipeline—from data extraction and leakage checks to final model training—has been documented and committed.

- **Codebase:** The full project, including all iterative notebooks, is available in the public repository: [Mahmoud-Beram/flyrank-ml-internship](https://github.com/Mahmoud-Beram/flyrank-ml-internship).
- **Execution Order:** The notebooks (`w01` through `w07`) are designed to be run sequentially.
- **Environment:** The analysis relies on `duckdb` for out-of-memory data processing and `scikit-learn` for modeling.
- **Determinism & Stability:** To ensure our results were robust and not a byproduct of a "lucky split," we verified model stability across 5 different random seeds. For the final pipeline and action queue generation, all algorithmic stochasticity was locked using a fixed seed (passed as `random_state = 42` in scikit-learn).
- **Data Access:** Re-running the pipeline requires a valid Hugging Face READ token set as `HF_TOKEN` in the environment secrets.

## 8. Acknowledgments & Data Credit

This analysis was conducted using the **FlyRank Internship Warehouse**. We acknowledge and thank FlyRank for providing the anonymized, aggregated Google Search Console (GSC) and content property dataset via Hugging Face (`hf://datasets/FlyRank/internship-warehouse`), which made this research possible.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.